In [0]:
/*
=============================================================================
PURPOSE: Maturity Dimension Table for Treasury Instruments
=============================================================================
This view provides a master reference table of treasury maturity definitions,
including descriptions, labels, and currency information.

TARGET SCHEMA: thekitchen.t (transformed data layer)
GRAIN: One row per maturity-country combination

BUSINESS VALUE:
- Provides human-readable descriptions for maturity keys used across fact tables
- Enables filtering and grouping by maturity characteristics
- Supports currency-aware analysis (extensible to multi-currency environments)
- Acts as a dimension table in star schema designs for yield curve analysis

CURRENT COVERAGE:
- US Treasury 2-Year, 5-Year, and 10-Year instruments
- USD currency only (can be extended to include other countries/currencies)

USAGE PATTERN:
- Join to yield fact tables on MaturityKey
- Use Description for display labels in dashboards and reports
- Filter by MaturityYears for maturity-specific analysis (e.g., short-term vs long-term)
- Group by Currency for multi-currency portfolio analysis
=============================================================================
*/

CREATE OR REPLACE VIEW t.maturity_tmp1 As
-- ============================================
-- Maturity Metadata for US Treasury Instruments
-- ============================================
-- Static reference data defining treasury maturity characteristics

-- 10-Year US Treasury
-- Long-term maturity, primary benchmark for economic health and mortgage rates
SELECT
  "10Y-US" AS MaturityKey,           -- Unique identifier: "10Y-US"
  "US Treasury 10Y" AS Description,  -- Human-readable full description
  "10Y" AS MaturityLabel,            -- Short display label for charts/reports
  10 AS MaturityYears,                -- Numeric maturity in years (for sorting/filtering)
  "USD" AS Currency                   -- Currency denomination

UNION ALL

-- 5-Year US Treasury
-- Mid-term maturity, influences corporate bonds and mortgage rates
SELECT
  "5Y-US" AS MaturityKey,            -- Unique identifier: "5Y-US"
  "US Treasury 5Y" AS Description,   -- Human-readable full description
  "5Y" AS MaturityLabel,             -- Short display label for charts/reports
  5 AS MaturityYears,                 -- Numeric maturity in years
  "USD" AS Currency                   -- Currency denomination

UNION ALL

-- 2-Year US Treasury
-- Short-term maturity, sensitive to Federal Reserve policy changes
SELECT
  "2Y-US" AS MaturityKey,            -- Unique identifier: "2Y-US"
  "US Treasury 2Y" AS Description,   -- Human-readable full description
  "2Y" AS MaturityLabel,             -- Short display label for charts/reports
  2 AS MaturityYears,                 -- Numeric maturity in years
  "USD" AS Currency                   -- Currency denomination

/*
=============================================================================
USAGE EXAMPLES:
=============================================================================

-- Example 1: List all available maturities with descriptions
SELECT
  MaturityKey,
  Description,
  MaturityLabel,
  MaturityYears,
  Currency
FROM t.maturity_tmp1
ORDER BY MaturityYears;

-- Example 2: Join to yield data for enriched reporting
SELECT
  y.Date,
  m.Description,
  m.MaturityLabel,
  y.Yield,
  m.Currency
FROM t.us_treasury_rates_tmp1 y
JOIN t.maturity_tmp1 m ON y.MaturityKey = m.MaturityKey
WHERE y.Date >= '2024-01-01'
ORDER BY y.Date DESC, m.MaturityYears;

-- Example 3: Filter for specific maturity categories
-- Short-term (< 3 years) vs Long-term (>= 10 years)
SELECT
  CASE
    WHEN MaturityYears < 3 THEN 'Short-Term'
    WHEN MaturityYears < 10 THEN 'Mid-Term'
    ELSE 'Long-Term'
  END AS maturity_category,
  COUNT(*) AS count,
  GROUP_CONCAT(MaturityLabel, ', ') AS maturities
FROM t.maturity_tmp1
GROUP BY maturity_category
ORDER BY MIN(MaturityYears);

-- Example 4: Calculate yield curve statistics by maturity
WITH latest_yields AS (
  SELECT
    y.MaturityKey,
    m.Description,
    m.MaturityYears,
    y.Yield
  FROM t.us_treasury_rates_tmp1 y
  JOIN t.maturity_tmp1 m ON y.MaturityKey = m.MaturityKey
  WHERE y.Date = (SELECT MAX(Date) FROM t.us_treasury_rates_tmp1)
)
SELECT
  Description,
  MaturityYears,
  ROUND(Yield, 4) AS current_yield,
  ROUND(Yield * 100, 2) AS yield_percentage
FROM latest_yields
ORDER BY MaturityYears;
=============================================================================
*/